##**Import Dependencies**


In [ ]:
'''!pip install deep-translator
'''

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import re
from collections import Counter
import seaborn as sns

In [ ]:
df = pd.read_csv('/content/multilingual_mobile_app_reviews_2025.csv')

In [ ]:
df.head()

In [ ]:
df.drop(columns=['review_id','user_id','app_version'], inplace=True)

In [ ]:
df.info()

In [ ]:
df.describe()

In [ ]:
#Detecting Outliers
cols = ['rating', 'num_helpful_votes', 'user_age']

outliers_dict = {}

for col in cols:
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1

    outliers = df[(df[col] < (Q1 - 1.5 * IQR)) | (df[col] > (Q3 + 1.5 * IQR))]

    print(f"\nColumn: {col}")
    print(f"Number of outliers: {len(outliers)}")

    if len(outliers) > 0:
        print(outliers[[col]].head())
    else:
        print("No outliers found ")

plt.figure(figsize=(12, 4))

for i, col in enumerate(cols, 1):
    plt.subplot(1, 3, i)
    plt.boxplot(df[col].dropna())
    plt.title(f"Boxplot of {col}")
plt.tight_layout()
plt.show()


##**Cleaning Data & Preparation**

In [ ]:
df.isnull().sum()

In [ ]:
df.duplicated().sum()

In [ ]:
df.dropna(subset=['review_text'], inplace=True)

In [ ]:
# Fill missing values in 'rating' column:
# For each group of (app_name, user_age), replace NaN with the group's mean rating
# If the group has no valid values, use the group's median instead
df['rating'] = df.groupby(['app_name','user_age'])['rating'].transform(
    lambda x: x.fillna(
        x.mean() if not x.dropna().empty else x.median()
    )
)

In [ ]:
# Fill missing values in 'rating' column:
# For each group of (app_name, verified_purchase), replace NaN with the group's mean rating.
# If the group has no valid values, use the group's mode instead.
df['rating'] = df.groupby(['app_name','verified_purchase'])['rating'].transform(
    lambda x: x.fillna(
        x.mean() if not x.dropna().empty
        else (x.mode()[0])
    )
)

In [ ]:
# Fill missing values in 'User_Country' column:
# For each group of (app_category, verified_purchase), replace NaN with the group's mode rating.
df['user_country']= df.groupby(['app_category','verified_purchase'])['user_country'].transform(
    lambda x: x.fillna(
        x.mode()[0]
    )
)

In [ ]:
#Filled the missing of the user_gender with Missing so we dont have a lot of bias in our data because it was already balanced between the categories we had about 500 for each and the missing was around 500 alone
df['user_gender'] = df['user_gender'].fillna('Missing')

In [ ]:
df['app_category'].value_counts().head(10)

In [ ]:
df['review_language'].value_counts()

In [ ]:
df['device_type'].value_counts()

In [ ]:
df['user_country'].value_counts().head(10)

In [ ]:
df['user_gender'].value_counts()

In [ ]:
df.isnull().sum()

In [ ]:
df['review_language'].value_counts()

##**Feature Engineering**

In [ ]:
#Did feature engineering in the data col so we can do a time series visulas later on
df['review_date'] = pd.to_datetime(df['review_date'])

In [ ]:
df['review_month_number']=df['review_date'].dt.month
df['review_month_name']=df['review_date'].dt.month_name()
df['review_year']=df['review_date'].dt.year

In [ ]:
df.info()

##**Text Preprocessing**

In [ ]:
#Translated the multilingual data we have to En so we can make all in one text preprocessing
'''
from deep_translator import GoogleTranslator


df["review_text_en"] = df["review_text"].apply(
    lambda x: GoogleTranslator(source="auto", target="en").translate(x) if pd.notnull(x) else x
)
'''

In [ ]:
df['review_text_en']

In [ ]:
#Did some NLP Text PreProcessing For the whole text(Convert to string_LowerCase_Remove Punctuation and Special Chracters_RemoveNumbers_Remove Whitespace_Removed the less than 2 characters words_Removed the stop words)
def clean_text_advanced(text):

    if pd.isna(text):
        return ""
    text = str(text).lower()

    text = re.sub(r'[^\w\s]', ' ', text)

    text = re.sub(r'\d+', ' ', text)

    text = ' '.join(text.split())

    words = [word for word in text.split() if len(word) >= 2]

    return ' '.join(words)

multilingual_stop_words = {

    'the', 'and', 'or', 'but', 'in', 'on', 'at', 'to', 'for', 'of', 'with', 'by', 'is', 'are', 'was', 'were', 'be', 'been', 'have', 'has', 'had', 'do', 'does', 'did', 'will', 'would', 'could', 'should', 'may', 'might', 'can', 'this', 'that', 'these', 'those', 'it', 'its', 'he', 'she', 'they', 'we', 'you', 'me', 'him', 'her', 'them', 'us', 'my', 'your', 'his', 'her', 'our', 'their', 'not', 'no', 'yes', 'all', 'any', 'some', 'more', 'most', 'very', 'too', 'so', 'just', 'now', 'then', 'here', 'there', 'when', 'where', 'how', 'why', 'what', 'who', 'which', 'if', 'an', 'as', 'up', 'out', 'get', 'go', 'see', 'know', 'one', 'two', 'first', 'last',

}

def remove_stop_words(text):

    if not text:
        return ""

    words = text.split()
    filtered_words = [word for word in words if word not in multilingual_stop_words]
    return ' '.join(filtered_words)

In [ ]:
df['review_text_clean'] = df['review_text_en'].apply(clean_text_advanced)
df['review_text_final'] = df['review_text_clean'].apply(remove_stop_words)

In [ ]:
#Comparisson Between the Process of cleaning the text
print("   BEFORE/AFTER CLEANING EXAMPLES:")
for i in range(5):
    original = df.iloc[i]['review_text']
    basic_clean = df.iloc[i]['review_text_clean']
    final_clean = df.iloc[i]['review_text_final']
    print(f"   Original:    '{original}'")
    print(f"   Basic Clean: '{basic_clean}'")
    print(f"   Final Clean: '{final_clean}'")
    print()

In [ ]:
#Categorsized the Ratings into 3 Categories
def categorize_sentiment(rating):

    if pd.isna(rating):
        return 'unknown'
    elif rating <= 2:
        return 'negative'
    elif rating >= 4:
        return 'positive'
    else:
        return 'neutral'

df['sentiment_category'] = df['rating'].apply(categorize_sentiment)


In [ ]:
sentiment_counts = df['sentiment_category'].value_counts()
sentiment_counts

In [ ]:
df[['app_name', 'rating', 'sentiment_category','review_text_final']].head()

In [ ]:
#Split the whole text into indivisual words
def get_all_words(text_series):

    all_words = []
    for text in text_series.dropna():
        if isinstance(text, str) and text.strip():
            words = [word for word in text.split() if len(word) >= 2]
            all_words.extend(words)
    return all_words

In [ ]:
all_review_words = get_all_words(df['review_text_final'])

In [ ]:
word_counter = Counter(all_review_words)
most_common_words = word_counter.most_common(20)
for i, (word, count) in enumerate(most_common_words, 1):
    percentage = (count / len(all_review_words)) * 100

In [ ]:
#Get word pairs (bigram)
def find_word_pairs(text_series):

    word_pairs = []
    for text in text_series.dropna():
        if isinstance(text, str) and text.strip():
            words = text.split()
            if len(words) >= 2:

                for i in range(len(words) - 1):
                    if len(words[i]) >= 2 and len(words[i+1]) >= 2:
                        pair = f"{words[i]} {words[i+1]}"
                        word_pairs.append(pair)
    return word_pairs

In [ ]:
word_pairs = find_word_pairs(df['review_text_final'])
pair_counter = Counter(word_pairs)
top_topics = pair_counter.most_common(15)


##**Visualizations**

In [ ]:
plt.style.use('default')
sns.set_palette("husl")
fig_size = (12, 8)

In [ ]:
#Ratings By Year
if len(df['review_year'].unique()) > 1:
    plt.figure(figsize=(12, 8))

    yearly_ratings = df.groupby('review_year')['rating'].agg(['mean', 'count']).reset_index()

    plt.plot(yearly_ratings['review_year'], yearly_ratings['mean'],
             marker='o', linewidth=2, markersize=8, color='red')
    plt.ylabel('Average Rating', fontsize=12)
    plt.xlabel('Year', fontsize=12)
    plt.title('Average App Ratings by Year', fontsize=14, fontweight='bold')
    plt.grid(True, alpha=0.3)
    plt.ylim(0, 5)


    for _, row in yearly_ratings.iterrows():
        plt.text(row['review_year'], row['mean'] + 0.1,
                 f"{row['mean']:.2f}\n({int(row['count'])})",
                 ha='center', fontsize=10)

    plt.tight_layout()
    plt.show()

    print("   Yearly Rating Statistics:")
    for _, row in yearly_ratings.iterrows():
        print(f"   {int(row['review_year'])}: {row['mean']:.2f} stars ({int(row['count'])} reviews)")
else:
    print("   Only one year of data available - skipping yearly trend analysis")

In [ ]:
#Ratings Trends for the Top 5 Apps
top_apps = df['app_name'].value_counts().head(5).index

plt.figure(figsize=(15, 10))


fig, axes = plt.subplots(2, 3, figsize=(18, 12))
axes = axes.flatten()

for i, app in enumerate(top_apps):
    app_data = df[df['app_name'] == app]


    app_monthly = app_data.groupby('review_month_number')['rating'].agg(['mean', 'count']).reset_index()
    app_monthly['month_name'] = app_monthly['review_month_number'].map({
        1: 'Jan', 2: 'Feb', 3: 'Mar', 4: 'Apr', 5: 'May', 6: 'Jun',
        7: 'Jul', 8: 'Aug', 9: 'Sep', 10: 'Oct', 11: 'Nov', 12: 'Dec'
    })


    axes[i].plot(app_monthly['review_month_number'], app_monthly['mean'],
                 marker='o', linewidth=2, markersize=6)
    axes[i].set_title(f'{app}\n({len(app_data)} reviews)', fontweight='bold')
    axes[i].set_ylabel('Rating')
    axes[i].set_xlabel('Month')
    axes[i].grid(True, alpha=0.3)
    axes[i].set_ylim(0, 5)
    axes[i].set_xticks(app_monthly['review_month_number'])
    axes[i].set_xticklabels([name[:3] for name in app_monthly['month_name']], rotation=45)


if len(top_apps) < 6:
    axes[5].set_visible(False)

plt.suptitle('Rating Trends for Top 5 Apps by Month', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
#AVG Rating by Season
def get_season(month):
    if month in [12, 1, 2]:
        return 'Winter'
    elif month in [3, 4, 5]:
        return 'Spring'
    elif month in [6, 7, 8]:
        return 'Summer'
    else:
        return 'Fall'

df['season'] = df['review_month_number'].apply(get_season)


seasonal_stats = df.groupby('season').agg({
    'rating': ['mean', 'count'],
    'review_month_number': 'first'
}).round(2)
seasonal_stats.columns = ['avg_rating', 'review_count', 'month_ref']
seasonal_stats = seasonal_stats.sort_values('month_ref')

plt.figure(figsize=(10, 6))
bars = plt.bar(seasonal_stats.index, seasonal_stats['avg_rating'],
               color=['lightblue', 'lightgreen', 'orange', 'brown'], alpha=0.7)
plt.ylabel('Average Rating', fontsize=12)
plt.xlabel('Season', fontsize=12)
plt.title('Average App Ratings by Season', fontsize=14, fontweight='bold')
plt.ylim(0, 5)
plt.grid(axis='y', alpha=0.3)


for i, bar in enumerate(bars):
    height = bar.get_height()
    count = seasonal_stats.iloc[i]['review_count']
    plt.text(bar.get_x() + bar.get_width()/2., height + 0.05,
             f'{height:.2f}\n({int(count)} reviews)',
             ha='center', va='bottom', fontsize=10)

plt.tight_layout()
plt.show()

In [ ]:
#AVG Ratings by Gender
plt.figure(figsize=(10, 6))


gender_ratings = df.groupby('user_gender')['rating'].agg(['mean', 'count', 'std']).round(3)

bars = plt.bar(gender_ratings.index, gender_ratings['mean'],
               color=['lightcoral', 'lightblue', 'lightgreen','skyblue','pink'][:len(gender_ratings)],
               alpha=0.7, edgecolor='black')


plt.errorbar(range(len(gender_ratings)), gender_ratings['mean'],
             yerr=gender_ratings['std'], fmt='none', color='black', capsize=5)

plt.ylabel('Average Rating', fontsize=12)
plt.xlabel('Gender', fontsize=12)
plt.title('Average App Ratings by Gender', fontsize=14, fontweight='bold')
plt.ylim(0, 5)
plt.grid(axis='y', alpha=0.3)


for i, (gender, data) in enumerate(gender_ratings.iterrows()):
    plt.text(i, data['mean'] + 0.1,
             f"{data['mean']:.2f}\n({int(data['count'])} reviews)",
             ha='center', fontsize=10)

plt.tight_layout()
plt.show()

print("   Rating statistics by gender:")
for gender, data in gender_ratings.iterrows():
    print(f"   {gender}: {data['mean']:.2f} ± {data['std']:.2f} stars ({int(data['count'])} reviews)")

In [ ]:
#Sentiment Distribution by Gender
if 'sentiment_category' not in df.columns:
    def categorize_sentiment(rating):
        if pd.isna(rating):
            return 'unknown'
        elif rating <= 2:
            return 'negative'
        elif rating >= 4:
            return 'positive'
        else:
            return 'neutral'
    df['sentiment_category'] = df['rating'].apply(categorize_sentiment)


sentiment_crosstab = pd.crosstab(df['user_gender'], df['sentiment_category'], normalize='index') * 100

plt.figure(figsize=(12, 6))
sentiment_crosstab.plot(kind='bar', ax=plt.gca(), color=['lightcoral', 'gold', 'lightgreen'])
plt.ylabel('Percentage (%)', fontsize=12)
plt.xlabel('Gender', fontsize=12)
plt.title('Sentiment Distribution by Gender (%)', fontsize=14, fontweight='bold')
plt.legend(title='Sentiment', bbox_to_anchor=(1.05, 1), loc='upper left')
plt.xticks(rotation=0)
plt.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

print("   Sentiment percentages by gender:")
for gender in sentiment_crosstab.index:
    print(f"   {gender}:")
    for sentiment in sentiment_crosstab.columns:
        print(f"      {sentiment}: {sentiment_crosstab.loc[gender, sentiment]:.1f}%")

In [ ]:
#App Preferences by Gender
plt.figure(figsize=(14, 8))

category_gender_counts = pd.crosstab(df['app_category'],
                                    df['user_gender'])

category_gender_counts.plot(kind='bar', ax=plt.gca(),
                           color=['lightcoral', 'lightblue', 'lightgreen','Blue','green'],
                           alpha=0.8, edgecolor='black')
plt.title('App Category Preferences by Gender\n(Review Counts)', fontsize=14, fontweight='bold')
plt.ylabel('Number of Reviews', fontsize=12)
plt.xlabel('App Category', fontsize=12)
plt.xticks(rotation=45, ha='right')
plt.legend(title='Gender', bbox_to_anchor=(1.05, 1), loc='upper left')
plt.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
#AVG User Age per APPS
plt.figure(figsize=(12,6))
sns.barplot(x="app_name", y="user_age", data=df, estimator="mean", errorbar=None, color="skyblue")
plt.xticks(rotation=45, ha="right")
plt.title("Average User Age per App")
plt.show()

In [ ]:
#AVG User Age by Category
plt.figure(figsize=(13,7))
sns.barplot(x="app_category", y="user_age", data=df, estimator="mean", errorbar=None , color="skyblue")
plt.xticks(rotation=45, ha="right")
plt.title("Average User Age per App Category")
plt.show()

In [ ]:
#Top 10 Apps By Device
top_apps = df['app_name'].value_counts().nlargest(10).index
subset = df[df['app_name'].isin(top_apps)]

plt.figure(figsize=(10,6))
sns.countplot(x="app_name", hue="device_type", data=subset)
plt.xticks(rotation=30, ha="right")
plt.title("Top 10 Apps by Device Type")
plt.show()


In [ ]:
#Top 20 Most Common Words in Reviews
all_words = []
for text in df['review_text_final'].dropna():
    if isinstance(text, str) and text.strip():
        words = [word for word in text.split() if len(word) >= 2]
        all_words.extend(words)

word_counter = Counter(all_words)
top_20_words = word_counter.most_common(20)
plt.figure(figsize=fig_size)
words = [word for word, count in top_20_words]
counts = [count for word, count in top_20_words]

plt.barh(words[::-1], counts[::-1], color='skyblue', edgecolor='navy', alpha=0.7)
plt.xlabel('Frequency', fontsize=12)
plt.ylabel('Words', fontsize=12)
plt.title('Top 20 Most Common Words in App Reviews)', fontsize=14, fontweight='bold')
plt.grid(axis='x', alpha=0.3)
for i, count in enumerate(counts[::-1]):
    plt.text(count + 0.5, i, str(count), va='center', fontsize=10)

plt.tight_layout()
plt.show()

In [ ]:
#Distribution of Review Sentiments
plt.figure(figsize=(10, 6))
sentiment_counts = df['sentiment_category'].value_counts()

colors = ['lightcoral', 'gold', 'lightgreen']
plt.pie(sentiment_counts.values, labels=sentiment_counts.index, autopct='%1.1f%%',
        colors=colors, startangle=90, textprops={'fontsize': 12})
plt.title('Distribution of Review Sentiments\n(Based on Ratings)', fontsize=14, fontweight='bold')
plt.axis('equal')
plt.show()

print(f"   Sentiment breakdown:")
for sentiment, count in sentiment_counts.items():
    percentage = (count / len(df)) * 100
    print(f"   {sentiment.capitalize()}: {count} reviews ({percentage:.1f}%)")

In [ ]:
#Top 10 Categories in use
plt.figure(figsize=fig_size)
top_categories = df['app_category'].value_counts().head(10)

top_categories = top_categories.sort_values(ascending=True)

plt.barh(range(len(top_categories)), top_categories.values, color='lightcoral', alpha=0.7)
plt.yticks(range(len(top_categories)), top_categories.index)
plt.xlabel('Number of Reviews', fontsize=12)
plt.ylabel('App Category', fontsize=12)
plt.title('Top 10 App Categories by Review Count', fontsize=14, fontweight='bold')
plt.grid(axis='x', alpha=0.3)

for i, count in enumerate(top_categories.values):
    plt.text(count + 1, i, str(count), va='center', fontsize=10)

plt.tight_layout()
plt.show()


In [ ]:
#Top Languages Count that uses the App
plt.figure(figsize=fig_size)
top_languages = df['review_language'].value_counts().head(12)

plt.bar(range(len(top_languages)), top_languages.values, color='lightgreen', alpha=0.7)
plt.xticks(range(len(top_languages)), top_languages.index, rotation=45)
plt.xlabel('Language', fontsize=12)
plt.ylabel('Number of Reviews', fontsize=12)
plt.title('Top Languages in App Reviews', fontsize=14, fontweight='bold')
plt.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
#The Top 15 Countries that are using the App
plt.figure(figsize=(14, 8))
top_countries = df['user_country'].value_counts().head(15)

top_countries = top_countries.sort_values(ascending=True)

plt.barh(range(len(top_countries)), top_countries.values,
         color='lightblue', edgecolor='navy', alpha=0.8)

plt.yticks(range(len(top_countries)), top_countries.index)
plt.xlabel('Number of Reviews', fontsize=12)
plt.ylabel('Country', fontsize=12)
plt.title('Top 15 Countries by App Review', fontsize=14, fontweight='bold')
plt.grid(axis='x', alpha=0.3)

for i, count in enumerate(top_countries.values):
    plt.text(count + 0.5, i, str(count), va='center', fontsize=10)

plt.tight_layout()
plt.show()


In [ ]:
#AVG Ratings by Country
country_ratings = df.groupby('user_country').agg({
    'rating': ['mean', 'count']
}).round(2)
country_ratings.columns = ['avg_rating', 'review_count']
country_ratings = country_ratings[country_ratings['review_count'] >= 10].sort_values('avg_rating', ascending=False)

plt.figure(figsize=(14, 8))
countries_to_plot = country_ratings.head(12)

bars = plt.bar(range(len(countries_to_plot)), countries_to_plot['avg_rating'],
               color='orange', alpha=0.7, edgecolor='darkred')
plt.xticks(range(len(countries_to_plot)), countries_to_plot.index, rotation=45, ha='right')
plt.ylabel('Average Rating', fontsize=12)
plt.xlabel('Country', fontsize=12)
plt.title('Average App Ratings by Country\n(Countries with 10+ reviews)', fontsize=14, fontweight='bold')
plt.ylim(0, 5)
plt.grid(axis='y', alpha=0.3)

for i, (country, data) in enumerate(countries_to_plot.iterrows()):
    plt.text(i, data['avg_rating'] + 0.05, f"{data['avg_rating']:.2f}",
             ha='center', fontsize=10)
    plt.text(i, 0.1, f"({int(data['review_count'])})",
             ha='center', fontsize=8, alpha=0.7)

plt.tight_layout()
plt.show()

In [ ]:
#Sentiment Distibution By The top 8 Countries
top_8_countries = df['user_country'].value_counts().head(8).index

fig, axes = plt.subplots(2, 4, figsize=(16, 8))
axes = axes.flatten()

for i, country in enumerate(top_8_countries):
    country_data = df[df['user_country'] == country]
    sentiment_counts = country_data['sentiment_category'].value_counts()

    # Create pie chart for each country
    colors = ['lightcoral', 'gold', 'lightgreen']
    axes[i].pie(sentiment_counts.values, labels=sentiment_counts.index, autopct='%1.1f%%',
                colors=colors, startangle=90)
    axes[i].set_title(f'{country}\n({len(country_data)} reviews)', fontweight='bold')

plt.suptitle('Sentiment Distribution by Top Countries', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
#Top word Pairs(bigram) For Sentiments

colors = {
    "positive": "seagreen",
    "neutral": "steelblue",
    "negative": "firebrick"
}


sentiment_pairs = {}
for sentiment, group in df.groupby("sentiment_category"):
    pairs = find_word_pairs(group["review_text_final"])
    sentiment_pairs[sentiment.lower()] = Counter(pairs).most_common(10)


order = ["positive", "neutral", "negative"]


for sentiment in order:
    if sentiment in sentiment_pairs:
        pairs, counts = zip(*sentiment_pairs[sentiment])
        plt.figure(figsize=(10,5))
        plt.barh(
            pairs[::-1], counts[::-1],
            alpha=0.7,
            color=colors.get(sentiment, "gray")
        )
        plt.title(f"Top Word Pairs for {sentiment.capitalize()} Reviews", fontsize=14)
        plt.xlabel("Frequency")
        plt.show()


##**Conclusion & Results**

In [ ]:
#The key Results for the top 5 Categories
top_categories = df['app_category'].value_counts().head(5)
category_insights = {}

for category in top_categories.index:
    category_reviews = df[df['app_category'] == category]

    avg_rating = category_reviews['rating'].mean()

    category_words = []
    for text in category_reviews['review_text_final'].dropna():
        if isinstance(text, str) and text.strip():
            category_words.extend(text.split())

    top_words = Counter(category_words).most_common(5)

    category_insights[category] = {
        'review_count': len(category_reviews),
        'avg_rating': avg_rating,
        'top_words': top_words
    }

for category, insights in category_insights.items():
    print(f"\n    {category.upper()}:")
    print(f"      Reviews: {insights['review_count']}")
    print(f"      Avg Rating: {insights['avg_rating']:.2f}/5")
    print(f"      Key Words: {', '.join([word for word, count in insights['top_words']])}")

In [ ]:
#The Key Results for every Device Type
device_analysis = {}
for device in df['device_type'].value_counts().head(3).index:
    device_reviews = df[df['device_type'] == device]

    device_analysis[device] = {
        'count': len(device_reviews),
        'avg_rating': device_reviews['rating'].mean(),
        'sentiment_dist': device_reviews['sentiment_category'].value_counts().to_dict()
    }

for device, analysis in device_analysis.items():
    print(f"\n    {device.upper()}:")
    print(f"      Review Count: {analysis['count']}")
    print(f"      Average Rating: {analysis['avg_rating']:.2f}/5")
    print(f"      Sentiment: {analysis['sentiment_dist']}")

In [ ]:
#The Key Results For The top 3 Language Used
top_3_languages = df['review_language'].value_counts().head(3)
for lang in top_3_languages.index:
    lang_reviews = df[df['review_language'] == lang]
    avg_rating = lang_reviews['rating'].mean()
    sentiment_dist = lang_reviews['sentiment_category'].value_counts()

    print(f"    {lang.upper()} ({len(lang_reviews)} reviews):")
    print(f"      Average Rating: {avg_rating:.2f}/5")
    print(f"      Sentiment Distribution: {sentiment_dist.to_dict()}")

In [ ]:
#Key Insights And Conclusions About the whole Dataset
total_reviews = len(df)
avg_rating = df['rating'].mean()
most_common_category = df['app_category'].value_counts().index[0]
most_common_device = df['device_type'].value_counts().index[0]
most_common_language = df['review_language'].value_counts().index[0]
sentiment_pcts = (df['sentiment_category'].value_counts() / total_reviews * 100).round(1)

print(f" DATASET OVERVIEW:")
print(f"   • Total Reviews Analyzed: {total_reviews:,}")
print(f"   • Average Rating: {avg_rating:.2f}/5 stars")
print(f"   • Most Common Category: {most_common_category}")
print(f"   • Most Common Device: {most_common_device}")
print(f"   • Most Common Language: {most_common_language}")

print(f"\n SENTIMENT BREAKDOWN:")
for sentiment, pct in sentiment_pcts.items():
    print(f"   • {sentiment.capitalize()}: {pct}%")

print(f"\n TOP INSIGHTS:")
print(f"   • Most discussed topics: {', '.join([pair.replace(' ', '+') for pair, count in top_topics[:3]])}")
print(f"   • User satisfaction: {'Generally satisfied' if avg_rating >= 3.5 else 'Mixed feelings' if avg_rating >= 2.5 else 'Needs improvement'}")

print(f"\n WORD FREQUENCIES BY SENTIMENT:")
results = {}
for sentiment in ['positive', 'neutral', 'negative']:
    sentiment_reviews = df[df['sentiment_category'] == sentiment]['review_text_final']
    sentiment_words = get_all_words(sentiment_reviews)

    if sentiment_words:
        sentiment_counter = Counter(sentiment_words)
        top_words = sentiment_counter.most_common(10)
        results[sentiment] = top_words

        print(f"\n    {sentiment.upper()} REVIEWS (top 10 words):")
        for word, count in top_words:
            print(f"      {word:<12} {count:>3} times")
